# 🌫️ Air Quality Anomaly Detection using Isolation Forest

**Dataset:** UCI Air Quality Dataset  
**Algorithm:** Isolation Forest (Unsupervised Anomaly Detection)  
**Goal:** Identify unusual / anomalous air quality readings that deviate from normal patterns

---

## What is Anomaly Detection?
Anomaly detection means finding data points that are significantly different from the rest.  
In air quality, an anomaly could be a sudden spike in CO or NOx — possibly due to industrial accidents, heavy traffic, or sensor errors.

## What is Isolation Forest?
Isolation Forest works by randomly splitting data. Anomalies are easier to isolate (need fewer splits), so they get lower anomaly scores. It doesn't need labelled data — it learns what's "normal" on its own.

---
## Notebook Flow
1. Import Libraries
2. Load Dataset
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Train Isolation Forest
6. Visualize Anomalies
7. Conclusion

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

# Make plots look clean
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

print('All libraries imported successfully!')

---
## 2. Load Dataset

The UCI Air Quality dataset is a `.csv` file (originally `.xlsx`, but we'll use the CSV version).  
Download it from: https://archive.ics.uci.edu/ml/datasets/Air+Quality

> **Note:** The dataset uses `;` as separator and `,` as decimal (European format). We handle that below.

**Key columns we'll use:**
| Column | Meaning |
|---|---|
| CO(GT) | True hourly CO concentration (mg/m³) |
| C6H6(GT) | True hourly Benzene concentration (µg/m³) |
| NOx(GT) | True hourly NOx concentration (ppb) |
| NO2(GT) | True hourly NO2 concentration (µg/m³) |
| T | Temperature (°C) |
| RH | Relative Humidity (%) |
| AH | Absolute Humidity |

In [ ]:
# Load the dataset
# The file uses semicolon separator and comma as decimal point
df = pd.read_csv('AirQualityUCI.csv', sep=';', decimal=',')

# Preview
print('Shape:', df.shape)
df.head()

In [ ]:
# The last two columns are often empty artifacts from Excel export — drop them
df = df.dropna(axis=1, how='all')

# Also drop empty rows at the end
df = df.dropna(axis=0, how='all')

print('Shape after dropping empty rows/cols:', df.shape)
df.columns.tolist()

---
## 3. Exploratory Data Analysis (EDA)

EDA helps us understand the data before modelling.  
We will check: shape, data types, missing values, distributions, and correlations.

### 3.1 Basic Info

In [ ]:
df.info()

In [ ]:
# Statistical summary of numeric columns
df.describe()

### 3.2 Handle Missing Values

> ⚠️ **Important:** In this dataset, `-200` is used as the sentinel value for missing data (not NaN).  
> We replace all -200 values with NaN so pandas can handle them properly.

In [ ]:
# Replace -200 with NaN
df.replace(-200, np.nan, inplace=True)

# Count missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Visualize missing data
plt.figure(figsize=(14, 4))
sns.heatmap(df.isnull().T, cbar=False, cmap='Reds', yticklabels=True)
plt.title('Missing Value Heatmap (Red = Missing)', fontsize=13)
plt.tight_layout()
plt.show()

### 3.3 Select Features for Analysis

We pick the sensor columns that represent actual pollutant measurements.  
We drop columns with too many missing values (like `NMHC(GT)` which is >90% missing).

In [ ]:
# Select the key pollutant + environmental columns
features = ['CO(GT)', 'C6H6(GT)', 'NOx(GT)', 'NO2(GT)', 'T', 'RH', 'AH']

df_features = df[features].copy()

print('Selected features shape:', df_features.shape)
df_features.head()

### 3.4 Distribution of Each Feature

Histograms show us whether data is skewed, has outliers, or follows a normal distribution.

In [ ]:
df_features.hist(bins=40, figsize=(16, 10), color='steelblue', edgecolor='white')
plt.suptitle('Distribution of Air Quality Features', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### 3.5 Box Plots — Spotting Outliers

Box plots show the spread and highlight extreme values (potential anomalies).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].boxplot(df_features[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='lightblue'))
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('')

# Hide the unused subplot
axes[-1].set_visible(False)

plt.suptitle('Box Plots — Outlier Inspection', fontsize=14)
plt.tight_layout()
plt.show()

### 3.6 Correlation Heatmap

Correlation tells us which features move together.  
High correlation between pollutants is expected (e.g., CO and Benzene from traffic).

In [ ]:
plt.figure(figsize=(9, 6))
corr = df_features.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix of Air Quality Features', fontsize=13)
plt.tight_layout()
plt.show()

### 3.7 Time Series — CO Concentration Over Time

Visualising CO over time helps us see if there are sudden spikes (likely anomalies).

In [ ]:
# Combine Date and Time into a single datetime column
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], 
                                 format='%d/%m/%Y %H.%M.%S', errors='coerce')

# Plot CO over time
plt.figure(figsize=(15, 4))
plt.plot(df['Datetime'], df['CO(GT)'], color='tomato', linewidth=0.6, alpha=0.8)
plt.title('CO Concentration Over Time', fontsize=13)
plt.xlabel('Date')
plt.ylabel('CO (mg/m³)')
plt.tight_layout()
plt.show()

---
## 4. Data Preprocessing

Before feeding data to Isolation Forest, we need to:
1. Drop rows that still have NaN values
2. Scale features (so no single feature dominates due to its scale)

In [ ]:
# Drop rows with any remaining NaN
df_clean = df_features.dropna()

print(f'Rows before dropping NaN: {len(df_features)}')
print(f'Rows after dropping NaN:  {len(df_clean)}')
print(f'Rows dropped: {len(df_features) - len(df_clean)}')

In [ ]:
# Scale features using StandardScaler
# This transforms each column to have mean=0, std=1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean)

print('Scaled data shape:', X_scaled.shape)

---
## 5. Isolation Forest — Anomaly Detection

**How it works:**
- Builds many random decision trees
- Anomalies are isolated closer to the root of the tree (fewer splits needed)
- Returns:
  - `1` → Normal point
  - `-1` → Anomaly

**Key Parameter:**
- `contamination`: The expected proportion of anomalies in the data. We set it to `0.05` (5%).

In [ ]:
# Train Isolation Forest
iso_forest = IsolationForest(
    n_estimators=100,       # number of trees
    contamination=0.05,     # we expect ~5% of data to be anomalies
    random_state=42         # for reproducibility
)

iso_forest.fit(X_scaled)
print('Model trained!')

In [ ]:
# Get predictions and anomaly scores
df_clean = df_clean.copy()
df_clean['anomaly'] = iso_forest.predict(X_scaled)       # 1 = normal, -1 = anomaly
df_clean['anomaly_score'] = iso_forest.score_samples(X_scaled)  # lower = more anomalous

# Count anomalies vs normal
counts = df_clean['anomaly'].value_counts()
print(f"Normal points : {counts[1]}")
print(f"Anomalies     : {counts[-1]}")
print(f"Anomaly rate  : {counts[-1]/len(df_clean)*100:.2f}%")

---
## 6. Visualize Anomalies

### 6.1 Anomaly Score Distribution

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(df_clean['anomaly_score'], bins=50, color='steelblue', edgecolor='white')
plt.axvline(df_clean[df_clean['anomaly'] == -1]['anomaly_score'].max(), 
            color='red', linestyle='--', label='Anomaly Threshold')
plt.title('Anomaly Score Distribution', fontsize=13)
plt.xlabel('Anomaly Score (lower = more anomalous)')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

### 6.2 CO vs NOx — Anomalies Highlighted

In [ ]:
normal = df_clean[df_clean['anomaly'] == 1]
anomalies = df_clean[df_clean['anomaly'] == -1]

plt.figure(figsize=(10, 6))
plt.scatter(normal['CO(GT)'], normal['NOx(GT)'],
            c='steelblue', s=15, alpha=0.5, label='Normal')
plt.scatter(anomalies['CO(GT)'], anomalies['NOx(GT)'],
            c='red', s=40, alpha=0.8, label='Anomaly', marker='x')
plt.title('CO vs NOx — Anomaly Detection', fontsize=13)
plt.xlabel('CO (mg/m³)')
plt.ylabel('NOx (ppb)')
plt.legend()
plt.tight_layout()
plt.show()

### 6.3 CO Over Time — Anomalies Highlighted

In [ ]:
# Add datetime back for time series plot
df_clean_with_time = df_clean.copy()
df_clean_with_time.index = df_clean.index  # preserve original index

# Align datetime with cleaned df
df_clean_with_time['Datetime'] = df.loc[df_clean.index, 'Datetime'].values

normal_t = df_clean_with_time[df_clean_with_time['anomaly'] == 1]
anomaly_t = df_clean_with_time[df_clean_with_time['anomaly'] == -1]

plt.figure(figsize=(15, 4))
plt.plot(normal_t['Datetime'], normal_t['CO(GT)'],
         color='steelblue', linewidth=0.6, alpha=0.7, label='Normal')
plt.scatter(anomaly_t['Datetime'], anomaly_t['CO(GT)'],
            color='red', s=25, zorder=5, label='Anomaly')
plt.title('CO Concentration Over Time — Anomalies Marked in Red', fontsize=13)
plt.xlabel('Date')
plt.ylabel('CO (mg/m³)')
plt.legend()
plt.tight_layout()
plt.show()

### 6.4 Feature-wise Anomaly Comparison

Let's compare the average values of each feature in normal vs anomalous data points.

In [ ]:
comparison = df_clean.groupby('anomaly')[features].mean().T
comparison.columns = ['Anomaly', 'Normal']

comparison.plot(kind='bar', figsize=(12, 5), color=['tomato', 'steelblue'], edgecolor='white')
plt.title('Average Feature Values — Normal vs Anomaly', fontsize=13)
plt.xlabel('Feature')
plt.ylabel('Mean Value')
plt.xticks(rotation=30)
plt.legend()
plt.tight_layout()
plt.show()

print(comparison)

---
## 7. Sample Anomalous Records

Let's look at the top anomalies — those with the lowest anomaly scores.

In [ ]:
top_anomalies = df_clean[df_clean['anomaly'] == -1].sort_values('anomaly_score').head(10)
print('Top 10 most anomalous data points:')
top_anomalies[features + ['anomaly_score']]

---
## 8. Conclusion

### What we did:
- Loaded the UCI Air Quality dataset and handled its European CSV format
- Replaced `-200` sentinel values with `NaN` and dropped incomplete rows
- Performed EDA: distributions, box plots, correlation heatmap, and time series plots
- Scaled features using `StandardScaler`
- Trained `IsolationForest` with 5% contamination rate
- Detected and visualised anomalies across scatter plots and time series

### Key Findings:
- Anomalies tend to occur when CO and/or NOx values are unusually high — likely indicating heavy traffic events or sensor malfunctions
- The correlation between CO, Benzene, and NOx is strong in normal readings — anomalies break this pattern
- Isolation Forest is effective for this kind of unsupervised anomaly detection where we don't have labelled data

### Possible Improvements:
- Try different `contamination` values (0.01, 0.10) and compare results
- Use PCA to reduce features before plotting in 2D
- Compare with other algorithms like Local Outlier Factor (LOF) or One-Class SVM